# WanGP Character Consistency on RunPod

Use this after pushing the modified repo to your own Git and opening it in a RunPod Jupyter environment.

In [ ]:
from pathlib import Path
ROOT = Path('/workspace/Wan2GP-main')
assert ROOT.exists(), ROOT
ROOT

## Build a character settings manifest

Upload your reference images somewhere under the RunPod workspace and update the paths below.

In [ ]:
import importlib.util

helper_path = ROOT / 'plugins' / 'wan2gp-character-consistency' / 'character_pack.py'
spec = importlib.util.spec_from_file_location('wan2gp_character_pack', helper_path)
packs = importlib.util.module_from_spec(spec)
spec.loader.exec_module(packs)

refs = [
    '/workspace/refs/character_front.png',
    '/workspace/refs/character_body.png',
    '/workspace/refs/character_profile.png',
]

manifest = packs.build_manifest(
    character_name='my_character',
    mode='bernini_ingredients',
    identity_prompt='A consistent named character. Replace this with exact face, hair, body, wardrobe, colors, and persistent props.',
    negative_prompt='different person, face drift, changed hairstyle, changed age, inconsistent outfit, distorted hands',
    style_prompt='cinematic natural light, realistic texture, stable color grade',
    shot_prompts=[
        'Medium close-up, the character looks toward camera and smiles subtly.',
        'Wide shot, the same character walks through a quiet city street at dusk.'
    ],
    refs=refs,
    resolution='1280x720',
    video_length=121,
    seed=-1,
)

settings_path = packs.save_manifest_json('my_character', manifest)
settings_path

## Generate through WanGP's Python API

In [ ]:
from shared.api import init

session = init(
    root=ROOT,
    cli_args=['--attention', 'sage2', '--profile', '4'],
)

job = session.submit(settings_path)
result = job.result()
print('success:', result.success)
print('files:', result.generated_files)
print('errors:', [str(error) for error in result.errors])